# Step 2: Clean the Southwest data

Loads the monthly BTS files, keeps Southwest (WN), selects the delay, taxi, cancellation and flight columns, fixes data types, adds an `OnTime` flag and `TotalDelay`, removes impossible delay values, renames the columns, and exports the cleaned file. Written in Google Colab; change `path` to wherever the monthly CSVs are saved.

In [ ]:
# Reset the environment (optional but recommended)
import pandas as pd
import numpy as np
import glob

# Reset df if it exists
try:
    del df
except:
    pass


# Import Required Python Libraries

In [ ]:
import pandas as pd
import numpy as np
import glob


# Load All Monthly BTS CSV Files Into One **Dataset**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
path = "/content/drive/MyDrive/Raw Airline Data /"

# Find ALL BTS files
all_files = glob.glob(path + "T_ONTIME_REPORTING*.csv")

print("Files found:", all_files)   # <-- See if it finds all months

df_list = [pd.read_csv(f, low_memory=False) for f in all_files]

df = pd.concat(df_list, ignore_index=True)

print("Merged data shape:", df.shape)
df.head()


# Filter Data to Keep Only Southwest Airlines (WN)

In [ ]:
df = df[df['OP_UNIQUE_CARRIER'] == 'WN']
print("Southwest-only shape:", df.shape)
df.head()


Currently the dataset has 18 column from which not all of them is needed for analysis

**We will keep ONLY the columns that matter for:**

- Delay analysis

- On-time performance

- Cancellations

- High-level summary

- Clean Tableau visualizations

✔ This makes your dataset clean, lightweight, and easy to understand.

# Selecting Relevant Columns for Southwest On-Time Analysis

In [ ]:
cols_to_keep = [
    'YEAR', 'MONTH', 'DAY_OF_WEEK', 'FL_DATE',
    'OP_UNIQUE_CARRIER',
    'DEP_DELAY', 'DEP_DELAY_NEW', 'DEP_DEL15',
    'ARR_DELAY', 'ARR_DELAY_NEW', 'ARR_DEL15',
    'TAXI_OUT', 'TAXI_IN',
    'CANCELLED', 'DIVERTED',
    'FLIGHTS'
]

df = df[cols_to_keep]
print("Columns after selection:", df.columns.tolist())
df.head()


In [ ]:
df.columns

# Convert FL_DATE to Proper Datetime

In [ ]:
df['FL_DATE'] = pd.to_datetime(df['FL_DATE'], format='%m/%d/%Y %I:%M:%S %p', errors='coerce')

print(df['FL_DATE'].head())


# Convert Delay, Cancellation, and Flight Columns to Numeric

In [ ]:
numeric_cols = [
    'YEAR', 'MONTH', 'DAY_OF_WEEK',
    'DEP_DELAY', 'DEP_DELAY_NEW', 'DEP_DEL15',
    'ARR_DELAY', 'ARR_DELAY_NEW', 'ARR_DEL15',
    'TAXI_OUT', 'TAXI_IN',
    'CANCELLED', 'DIVERTED', 'FLIGHTS'
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df[numeric_cols].dtypes


# Create ON_TIME Indicator

In [ ]:
df['ON_TIME'] = (df['ARR_DELAY'] <= 0).astype(int)
df[['ARR_DELAY', 'ON_TIME']].head()


In [ ]:
df['ARR_DELAY_15'] = df['ARR_DEL15'].fillna(0).astype(int)
df[['ARR_DELAY', 'ARR_DELAY_15']].head()


In [ ]:
df['TOTAL_DELAY'] = df['DEP_DELAY_NEW'] + df['ARR_DELAY_NEW']
df[['DEP_DELAY_NEW', 'ARR_DELAY_NEW', 'TOTAL_DELAY']].head()


# Remove Invalid Delay Rows Safely

In [ ]:
df = df[df['MONTH'].between(1, 12)]     # valid months only
df = df[df['DEP_DELAY'] > -60]          # remove impossible early departures
df = df[df['ARR_DELAY'] > -60]          # remove impossible early arrivals

print("Rows after cleaning invalid values:", df.shape)
df.head()


# Give Column Names That Are Easy To Understand



In [ ]:
df = df.rename(columns={
    'YEAR': 'Year',
    'MONTH': 'Month',
    'DAY_OF_WEEK': 'DayOfWeek',
    'FL_DATE': 'FlightDate',
    'OP_UNIQUE_CARRIER': 'Airline',
    'DEP_DELAY': 'DepartureDelay',
    'DEP_DELAY_NEW': 'DepartureDelayNew',
    'DEP_DEL15': 'DepartureDelay15',
    'ARR_DELAY': 'ArrivalDelay',
    'ARR_DELAY_NEW': 'ArrivalDelayNew',
    'ARR_DEL15': 'ArrivalDelay15',
    'TAXI_OUT': 'TaxiOut',
    'TAXI_IN': 'TaxiIn',
    'CANCELLED': 'Cancelled',
    'DIVERTED': 'Diverted',
    'FLIGHTS': 'Flights',
    'ON_TIME': 'OnTime',
    'ARR_DELAY_15': 'ArrivalDelay15Flag',
    'TOTAL_DELAY': 'TotalDelay'
})

print(df.columns)
df.head()


In [ ]:
df = df.drop(columns=["ArrivalDelay15Flag"])

In [ ]:
df.columns

# Export Final Clean Southwest Airlines Dataset

In [ ]:
output_path = "/content/southwest_airlines_cleaned_data.csv"
df.to_csv(output_path, index=False)

print("File saved as:", output_path)


In [ ]:
df.head(20)


In [ ]:
df.dtypes


In [ ]:
df.describe(include='all')
